# 05. Valutazione e calibrazione

Un decision model restituisce probabilita, quindi l'accuratezza da sola non basta. Ci chiediamo anche se le probabilita sono calibrate: quando il modello dice 0.8, dovrebbe avere ragione circa l'80 percento delle volte?

Calcoliamo quattro cose: accuratezza, Brier score, errore di calibrazione atteso e una soglia di confidenza che passa oltre le decisioni incerte. Tutte le celle sono piccole e terminano con una validazione.


## Passo 1. Caricare il dataset e un client

Usiamo il mock cosi il notebook e riproducibile senza server. Il mock sbaglia la scelta nel 25 percento dei casi, cosi possiamo misurare accuratezza e calibrazione in modo onesto. In seguito sostituite con un client reale; il codice di valutazione non cambia.


In [ ]:
from decision_qa import MockClient, read_jsonl
records = read_jsonl('data/pubmedqa_decisions.jsonl')
print(len(records))


In [ ]:
client = MockClient(seed=42, quality=0.75, error_rate=0.25)
print(type(client).__name__)


## Passo 2. Un risultato, una probabilita

`gold_probability` estrae la probabilita che il modello ha assegnato alla risposta corretta. Lo mostriamo su una singola domanda prima di iterare.


In [ ]:
from decision_qa import gold_label, gold_probability, predicted_label
record = records[0]
question = record.questions[0]
result = client.decision(record.state, [question])[0]
print(result.probabilities)


In [ ]:
p = gold_probability(result, question)
print(round(p, 3))
assert 0.0 <= p <= 1.0


In [ ]:
print(gold_label(question))
print(predicted_label(result))


## Passo 3. Raccogliere tutte le previsioni

Scorriamo ogni record e ogni domanda. Per ogni risposta memorizziamo tre cose: la probabilita della risposta gold, l'etichetta scelta dal modello e la confidenza di quella etichetta. La calibrazione si calcola sempre sulla confidenza prevista, non su quella gold.


In [ ]:
gold_probs = []
confidences = []
correct = []
all_results = []
all_questions = []
for rec in records:
    results = client.decision(rec.state, rec.questions)
    all_results.extend(results)
    all_questions.extend(rec.questions)
    for q, res in zip(rec.questions, results):
        gold_probs.append(gold_probability(res, q))
        confidences.append(max(res.probabilities.values()))
        correct.append(int(predicted_label(res) == gold_label(q)))


In [ ]:
print(len(gold_probs), len(confidences), len(correct))
assert len(gold_probs) == len(confidences) == len(correct)


## Passo 4. Accuratezza

L'accuratezza e la frazione di domande in cui la risposta corretta era quella con probabilita massima. Non dice nulla sulla qualita dei valori di probabilita.


In [ ]:
acc = sum(correct) / len(correct)
print(round(acc, 3))
assert 0.0 <= acc <= 1.0


In [ ]:
from decision_qa import accuracy
library_acc = accuracy(all_results, all_questions)
print(round(library_acc, 3))
assert abs(library_acc - acc) < 1e-9


## Passo 5. Brier score

Il Brier score e l'errore quadratico medio delle probabilita gold. Piu basso e meglio. Un modello accurato e sicuro di se si avvicina a zero.


In [ ]:
from decision_qa import brier
score = brier(gold_probs)
print(round(score, 3))
assert 0.0 <= score <= 1.0


In [ ]:
one = brier([0.8])
print(round(one, 3))
assert abs(one - 0.04) < 1e-9


In [ ]:
from decision_qa import brier_multiclass
multiclass = brier_multiclass(all_results, all_questions)
print(round(multiclass, 3))
assert 0.0 <= multiclass <= 2.0


## Passo 6. Affidabilita ed ECE

La tabella di affidabilita raggruppa le previsioni per confidenza e confronta la confidenza media con l'accuratezza osservata. L'ECE e lo scarto pesato tra i due.


In [ ]:
from decision_qa import reliability
rows = reliability(confidences, correct)
print(len(rows))


In [ ]:
for row in rows:
    print(row['bin'], row['count'], round(row['confidence'], 2), round(row['accuracy'], 2))


Il diagramma di affidabilita mostra l'accuratezza osservata rispetto alla confidenza prevista. Ogni punto e un intervallo di confidenza e la sua dimensione mostra quanti casi contiene. I punti sulla diagonale tratteggiata sono perfettamente calibrati; i punti sotto la diagonale indicano un modello troppo sicuro.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.8, 3.4), constrained_layout=True)
ax.plot([0, 1], [0, 1], '--', color='gray', label='perfect calibration')
for index, row in enumerate(rows):
    offset = (8, -12) if index % 2 == 0 else (-8, 10)
    alignment = 'left' if index % 2 == 0 else 'right'
    ax.scatter(row['confidence'], row['accuracy'], s=20 + 3 * row['count'], color='#E45756', alpha=0.8, zorder=3)
    ax.annotate(f"n={row['count']}", (row['confidence'], row['accuracy']), textcoords='offset points', xytext=offset, ha=alignment, fontsize=8)
ax.set_xlabel('Predicted confidence')
ax.set_ylabel('Observed accuracy')
ax.set_title('Reliability diagram')
ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.08)
ax.grid(alpha=0.25)
ax.text(0.98, 0.06, 'marker size = count', ha='right', fontsize=8, color='#555555')
ax.legend(loc='upper left')
plt.show()


In [ ]:
from decision_qa import ece
error = ece(confidences, correct)
print(round(error, 3))
assert 0.0 <= error <= 1.0


## Passo 7. Una soglia di confidenza

`fit_threshold` trova il taglio piu basso che raggiunge una precisione target su questi dati. Le decisioni sotto il taglio vanno passate a un modello piu forte. In un progetto reale la soglia va calcolata su uno split di validazione, mai su quello di test.


In [ ]:
from decision_qa import fit_threshold
threshold = fit_threshold(confidences, correct, target_precision=0.9)
print(round(threshold, 2))


In [ ]:
coverage = sum(p >= threshold for p in confidences) / len(confidences)
print(round(coverage, 3))
assert 0.0 <= coverage <= 1.0


## Passo 8. Invarianza all'ordine delle opzioni

Infine verifichiamo che una choice etichettata mantenga la stessa risposta quando le opzioni vengono riordinate. Confrontiamo un modello affidabile con quello rumoroso: l'affidabile dovrebbe essere stabile, mentre il rumoroso puo fallire il controllo. Questo protegge dal bias posizionale.


In [ ]:
from decision_qa import ChoiceQuestion, MockClient
from decision_qa.evaluate import option_order_invariance
q = ChoiceQuestion(label='check', options=['a', 'b', 'c'], gold_index=1)
stable = MockClient(seed=5)


In [ ]:
print('stable model:', option_order_invariance(stable, {}, q))


In [ ]:
print('noisy model :', option_order_invariance(client, {}, q))


In [ ]:
assert option_order_invariance(stable, {}, q) is True
print('the reliable model is stable under reordering')


## Leggere i risultati con spirito critico

Alta accuratezza con scarsa calibrazione significa che le classifiche sono utili ma le probabilita non vanno fidate come numeri. Riferimenti gratuiti: JevBench sotto licenza MIT, ProbabilityBench, l'articolo Evaluating and Benchmarking the System One model, l'Early Evidence Audit con la sua checklist a 14 punti e Beyond Calibration. JevAdvBench ricorda che lo stato e input non attendibile, e Jev Does Not Play Dice mostra quanto le probabilita possano allontanarsi dalla realta.
